# Set Up Quality Gates for Your Support Bot in 20 Minutes

Describe what your LLM app does in plain text, get recommended eval metrics, and run them on your outputs.

| Time | Difficulty |
|------|----------|
| 20 min | Intermediate |

Your team just launched a RAG-based support bot for ReturnRight, an e-commerce platform. After the first week in production, customers started reporting wrong prices in responses and one user received an email address that belonged to someone else. You need quality gates, fast, but there are dozens of available eval metrics and you don't have time to research each one.

`AutoEvalPipeline` solves this by analyzing a plain-text description of your app and recommending the right eval metrics and safety scanners. You review what it picked, tune thresholds for your domain, run it on real outputs, and export the config for CI.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/auto-eval-pipeline.ipynb)

**Prerequisites**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see Get your API keys)
- Python 3.9+

In [ ]:
!pip install ai-evaluation

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"

## Step 1: Describe ReturnRight's support bot and get recommended evals

Write a plain-text description of what your app does. Be specific: mention whether it retrieves documents, generates content, uses tools, or handles sensitive data. The more context you give, the better the recommendations.

In [ ]:
from fi.evals.autoeval import AutoEvalPipeline

pipeline = AutoEvalPipeline.from_description(
    "A RAG-based customer support chatbot for an e-commerce platform. "
    "It retrieves product specs, return policies, and order history, "
    "then generates answers to customer questions. Customers sometimes "
    "share order IDs and email addresses in their messages."
)

print(pipeline.explain())

The pipeline analyzes your description and identifies:
- **App category** (e.g., RAG system, customer support, code assistant)
- **Risk level** based on the domain (general, healthcare, financial)
- **Domain sensitivity** (PII handling, compliance requirements)

From these, it selects evals that catch the failure modes your app is most likely to hit, and scanners that guard against safety risks.

## Step 2: Review what the pipeline selected

Before running anything, inspect the configuration. You want to understand exactly which evals and scanners were chosen and what thresholds they use.

In [ ]:
from fi.evals.autoeval import list_templates

# See the full config
print(pipeline.summary())
print()

# Inspect individual evaluations
for eval_cfg in pipeline.config.evaluations:
    print(f"  Eval: {eval_cfg.name:<25} threshold: {eval_cfg.threshold}  weight: {eval_cfg.weight}")

print()

# Inspect scanners
for scanner_cfg in pipeline.config.scanners:
    print(f"  Scanner: {scanner_cfg.name:<25} action: {scanner_cfg.action}  threshold: {scanner_cfg.threshold}")

print()

# See what other templates are available
print("Available templates:")
for name, desc in list_templates().items():
    print(f"  {name}: {desc}")

If the recommendations don't match your needs, you can start from a pre-built template instead:

In [ ]:
# Start from a template if the auto-detected category is wrong
rag_pipeline = AutoEvalPipeline.from_template("rag_system")
print(rag_pipeline.summary())

# Since ReturnRight is a RAG support bot, use the rag_system template
pipeline = AutoEvalPipeline.from_template("rag_system")
print(pipeline.summary())


Templates are available for `customer_support`, `rag_system`, `code_assistant`, `content_moderation`, `agent_workflow`, `healthcare`, and `financial`.

## Step 3: Tune the pipeline for ReturnRight's needs

The auto-generated pipeline is a starting point. Tune it by adjusting thresholds, adding evals you care about, or removing ones that don't apply.

In [ ]:
from fi.evals.autoeval import EvalConfig, ScannerConfig

# Tighten the faithfulness threshold (your app can't afford hallucinated prices)
pipeline.set_threshold("faithfulness", 0.85)

# Add completeness to catch incomplete answers
pipeline.add(EvalConfig(
    name="completeness",
    threshold=0.7,
    weight=1.0,
))

# Disable an eval temporarily without removing it
pipeline.disable("groundedness")

# Check the updated config
print(pipeline.summary())


Every change is chainable:

In [ ]:
pipeline.set_threshold("faithfulness", 0.85).add(
    EvalConfig(name="answer_relevancy", threshold=0.8)
).remove("InvisibleCharScanner")

## Step 4: Run the pipeline on ReturnRight's real outputs

Now evaluate actual outputs from your app. Pass a dictionary with the fields your evals expect (typically `query`, `response`, and `context` for RAG apps).

In [ ]:
test_cases = [
    {
        "query": "What's your return policy for electronics?",
        "response": (
            "You can return electronics within 30 days of purchase for a full refund, "
            "as long as the item is in its original packaging and unused."
        ),
        "context": (
            "Return policy: Electronics may be returned within 30 days of purchase "
            "for a full refund. Items must be unused and in original packaging. "
            "Opened software is non-refundable."
        ),
    },
    {
        "query": "Where is my order #12345?",
        "response": (
            "Your order #12345 shipped on March 10 and is expected to arrive by March 14. "
            "Here's your tracking number: 1Z999AA10123456784."
        ),
        "context": (
            "Order #12345: Shipped 2025-03-10 via UPS Ground. "
            "Tracking: 1Z999AA10123456784. Estimated delivery: 2025-03-14."
        ),
    },
    {
        "query": "Do you sell laptops with 64GB RAM?",
        "response": (
            "Yes! We carry several laptops with 64GB RAM. Our most popular is the "
            "ProBook X1 starting at $1,899, which also comes in a 128GB configuration."
        ),
        "context": (
            "ProBook X1: 64GB RAM, starts at $1,499. "
            "ProBook X1 Max: 128GB RAM, starts at $2,299. "
            "No other models offer 64GB or above."
        ),
    },
]

for i, tc in enumerate(test_cases):
    result = pipeline.evaluate(tc)
    status = "PASSED" if result.passed else "FAILED"
    print(f"\nCase {i+1}: {status} ({result.total_latency_ms:.0f}ms)")
    print(result.explain())

Case 3 contains a factual error: the response says the ProBook X1 starts at $1,899, but the context says $1,499. A well-configured pipeline catches this through faithfulness and groundedness checks.

The `result.explain()` method gives you a full breakdown of every eval and scanner that ran, including scores, pass/fail status, and which scanner (if any) blocked the response.

## Step 5: Find the pattern behind ReturnRight's failures

After running the pipeline, aggregate the results to find patterns. Are failures concentrated in one eval? That tells you exactly what to fix in your app.

In [ ]:
from collections import Counter

# Re-run all test cases to collect results for aggregation
all_results = [pipeline.evaluate(tc) for tc in test_cases]

pass_count = sum(1 for r in all_results if r.passed)
print(f"Overall: {pass_count}/{len(all_results)} passed\n")

# Check which metrics failed most often
metric_failures = Counter()
for r in all_results:
    for mr in r.metric_results:
        if not getattr(mr, "passed", True):
            metric_failures[getattr(mr, "eval_name", "unknown")] += 1

if metric_failures:
    print("Most common failures:")
    for metric, count in metric_failures.most_common():
        print(f"  {metric}: {count}/{len(all_results)} cases failed")
else:
    print("No metric failures detected.")

Common patterns and what they tell you:
- **Faithfulness failures**: your app is generating content not supported by the retrieved context. Fix your retrieval or tighten your system prompt.
- **Relevancy failures**: responses don't address the user's question. Check your query routing.
- **Scanner blocks**: the app is leaking PII or producing unsafe content. Add output filtering.

Once you've fixed the issues, re-run the pipeline to verify improvements.

## Step 6: Save the pipeline config for CI

Export your tuned pipeline so you can version-control it alongside your app code and reload it in CI or production.

In [ ]:
# Export to YAML (human-readable, good for code review)
pipeline.export_yaml("eval_pipeline.yaml")

# Export to JSON (good for programmatic use)
pipeline.export_json("eval_pipeline.json")

# Reload later
reloaded = AutoEvalPipeline.from_yaml("eval_pipeline.yaml")
print(reloaded.summary())

The YAML file captures every eval, scanner, threshold, and execution setting. Commit it to your repo so pipeline changes go through the same review process as code changes.

> **Tip:** For running this pipeline automatically on every PR, see CI/CD Eval Pipeline for the full GitHub Actions setup with quality gates and branch protection.

## What you solved

You took ReturnRight's support bot from zero quality gates to a tuned eval pipeline that catches the pricing bug and PII leak from week one, ready to run in CI on every deploy.

- **Described your app** in plain text and got a recommended set of evals and safety scanners
- **Reviewed and customized** the pipeline by adjusting thresholds, adding evals, and removing irrelevant scanners
- **Ran the pipeline** on real outputs and caught a factual error the app produced
- **Interpreted failures** to identify what to fix in your app
- **Exported the config** to YAML for version control and CI/CD reuse